# Urban100, full set: 384-px tiles plus per-image R2R adaptation with the frozen pipeline

The settings were chosen on the 12 Urban100 development images (`adaptation-dev12`, `urban100-tiles`) and confirmed
on the other 88 (`adaptation-confirm`):
* per-channel conv gains plus GroupNorm γ/β; Muon (AdamW branch) at lr 1e-3; 4 × 256² crops; 90 steps;
* R2R with α = 0.5, trained at t(σ√(1+α²)) and applied at t(σ);
* 384-px tiles with overlap 96; the final estimate is the horizontal-flip average (×2).

This run completes the 100-image numbers (amendment 7):
SEED-REPLICATION variant, σ 25: Urban100 confirm-88, with new noise draws and new R2R seeds.
* **dev-12 at σ 25 / 50,** with exactly the confirmed pipeline. These rows are descriptive only: the full-100 mean is
  confirm-88 (from `adaptation-confirm`) plus these 12, combined offline. Dev-12 chose the settings, so it is not
  tested here, and no SOTA flag is computed from it (it is easier than confirm-88).
* **All 100 images at σ 15.** The R2R adaptation was never tuned at σ 15, but dev-12's σ 15 scores were seen when the
  tile size was chosen, so the test is on confirm-88 only. Success: adapted ×2 − unadapted ×2 ≥ +0.08 dB, sign-flip
  p < 0.01, ≤ 5 % of images worse by more than 0.05.
* **No SOTA flag.** Published numbers are full-100 means, and our full-100 mean would include the 12 tuning images.

Noise: DMID's protocol with the seeds of `diffusion-tweedie-benchmark` (order McMaster, Kodak24, CBSD68, Urban100).
Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
from PIL import Image

ALPHA, LR, CROP, BATCH, STEPS = 0.5, 1e-3, 256, 4, 90
DEV12 = list(range(8, 100, 8))
SEED_SETS = ["McMaster", "Kodak24", "CBSD68", "Urban100"]
PLAN = {"Urban100": (None, None, "tiles-384", None)}   # set: (unused, unused, primary mode, secondary mode)
SEED_OFFSET = 2   # 0 is the published draw set; 1 and 2 replicate it
JOBS = [("Urban100", i, 25) for i in range(100) if i not in DEV12]
PUBLISHED = {("Urban100", 15): 35.70, ("Urban100", 25): 33.57, ("Urban100", 50): 30.64}   # PTG-RM
TIME_LIMIT_H = 11.5
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__, "steps", STEPS)

MUON_SRC = r'''"""Muon: momentum orthogonalised by Newton–Schulz, for hidden weight matrices.

Muon (Jordan et al., 2024) replaces the update of each weight matrix with the
nearest semi-orthogonal matrix to its Nesterov momentum, computed by five
quintic Newton–Schulz iterations; convolution kernels are flattened to
``(out, in · kh · kw)``. It is meant for hidden matrices only, so embeddings,
the output layer, biases and normalisation gains are updated by AdamW inside
the same optimizer, as in the reference implementation.

Muon updates are scaled by ``0.2 · sqrt(max(rows, columns))`` (Liu et al.,
2025, "Muon is Scalable for LLM Training"), which matches their RMS to AdamW's:
learning rates and weight decay tuned for AdamW carry over unchanged.
"""

from __future__ import annotations

import math
from typing import Iterable

import torch
import torch.nn as nn

NEWTON_SCHULZ = (3.4445, -4.7750, 2.0315)


def newton_schulz(matrix: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """Approximate the orthogonal factor ``U V^T`` of ``matrix = U S V^T``; singular values end up in about [0.7, 1.2]."""
    a, b, c = NEWTON_SCHULZ
    native_bf16 = matrix.is_cuda and torch.cuda.get_device_capability(matrix.device)[0] >= 8
    x = matrix.to(torch.bfloat16 if native_bf16 else torch.float32)
    tall = x.size(-2) > x.size(-1)
    if tall:
        x = x.mT
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        gram = x @ x.mT
        x = a * x + (b * gram + c * gram @ gram) @ x
    return (x.mT if tall else x).to(matrix.dtype)


def muon_parameter_groups(model: nn.Module, adamw_modules: Iterable[str] = ("head", "lm_head")) -> list[dict]:
    """Split ``model``'s parameters into a Muon group and an AdamW group.

    AdamW gets every parameter with fewer than two dimensions, every
    ``nn.Embedding`` weight (with any output layer tied to it) and every
    parameter of a module named in ``adamw_modules``, matched against the
    qualified name or its last component. Muon gets the remaining matrices.
    """
    names = set(adamw_modules)
    adamw = {
        id(p)
        for name, module in model.named_modules()
        if isinstance(module, nn.Embedding) or name in names or name.rsplit(".", 1)[-1] in names
        for p in module.parameters(recurse=False)
    }
    muon, other = [], []
    for p in model.parameters():  # tied weights appear once
        (other if p.ndim < 2 or id(p) in adamw else muon).append(p)
    return [{"params": muon, "use_muon": True}, {"params": other, "use_muon": False}]


class Muon(torch.optim.Optimizer):
    """Muon for hidden weight matrices, AdamW for every other parameter.

    ``params`` holds tensors, split by shape (two or more dimensions go to
    Muon), or groups with a ``use_muon`` flag such as
    :func:`muon_parameter_groups` returns. ``lr`` and decoupled
    ``weight_decay`` apply to both kinds; ``momentum``, ``nesterov`` and
    ``ns_steps`` to Muon; ``betas`` and ``eps`` to AdamW.
    """

    def __init__(
        self,
        params,
        lr: float = 3e-4,
        weight_decay: float = 0.0,
        momentum: float = 0.95,
        nesterov: bool = True,
        ns_steps: int = 5,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
    ):
        params = list(params)
        if params and not isinstance(params[0], dict):
            params = [
                {"params": [p for p in params if p.ndim >= 2], "use_muon": True},
                {"params": [p for p in params if p.ndim < 2], "use_muon": False},
            ]
        groups = [{**group, "params": list(group["params"])} for group in params]
        if any("use_muon" not in group for group in groups):
            raise ValueError("every parameter group needs a 'use_muon' flag")
        defaults = dict(
            lr=lr, weight_decay=weight_decay, momentum=momentum, nesterov=nesterov,
            ns_steps=ns_steps, betas=betas, eps=eps,
        )
        super().__init__([group for group in groups if group["params"]], defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            update = self._muon_update if group["use_muon"] else self._adamw_update
            for p in group["params"]:
                if p.grad is None:
                    continue
                direction = update(p.grad, self.state[p], group)
                p.mul_(1 - group["lr"] * group["weight_decay"])
                p.add_(direction, alpha=-group["lr"])
        return loss

    @staticmethod
    def _muon_update(grad, state, group):
        if not state:
            state["momentum_buffer"] = torch.zeros_like(grad)
        buffer = state["momentum_buffer"]
        buffer.lerp_(grad, 1 - group["momentum"])
        momentum = grad.lerp(buffer, group["momentum"]) if group["nesterov"] else buffer
        matrix = newton_schulz(momentum.reshape(len(momentum), -1), group["ns_steps"])
        return matrix.reshape_as(grad) * (0.2 * math.sqrt(max(matrix.shape)))

    @staticmethod
    def _adamw_update(grad, state, group):
        beta1, beta2 = group["betas"]
        if not state:
            state["step"] = 0
            state["exp_avg"] = torch.zeros_like(grad)
            state["exp_avg_sq"] = torch.zeros_like(grad)
        state["step"] += 1
        state["exp_avg"].lerp_(grad, 1 - beta1)
        state["exp_avg_sq"].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        corrected = (state["exp_avg_sq"] / (1 - beta2 ** state["step"])).sqrt().add_(group["eps"])
        return state["exp_avg"] / (1 - beta1 ** state["step"]) / corrected
'''  # laboratory/optimization/muon.py
_ns = {"__name__": "muon"}
exec(compile(MUON_SRC, "muon.py", "exec"), _ns)
Muon = _ns["Muon"]


def download(url, path, size=None):
    """Resumable download; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


t0 = time.time()
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": ("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", 367_018_356),
}
with ThreadPoolExecutor(2) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TEST = {}
with zipfile.ZipFile(P["datasets"]) as z:
    for ds in PLAN:
        names = sorted([n for n in z.namelist() if n.startswith(f"Datasets/Gaussion/{ds}/") and not n.endswith("/")], key=natkey)
        TEST[ds] = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))) for n in names]
TIMES["downloads"] = time.time() - t0
print({ds: len(v) for ds, v in TEST.items()}, f"downloads {TIMES['downloads'] / 60:.1f} min")


def noisy_of(ds, idx, sigma):
    """DMID's protocol with the seeds of diffusion-tweedie-benchmark: sample std exactly σ/255, unclipped."""
    seed = 1_000_000 * (SEED_SETS.index(ds) + 1 + 10 * SEED_OFFSET) + 1000 * sigma + idx
    x = torch.from_numpy(TEST[ds][idx][1]).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


def psnr(out, clean_u8):
    a = (out[0].clamp(0, 1) * 255).round().double()
    b = torch.from_numpy(clean_u8).permute(2, 0, 1).to(a.device).double()
    return 10 * math.log10(255 ** 2 / (a - b).pow(2).mean().item())

## 1 · ADM in fp32 weights with fp16 autocast and gradient checkpointing; per-channel conv gains + GroupNorm affine

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def checkpoint(func, inputs, params, flag):
    if flag and torch.is_grad_enabled():
        return torch.utils.checkpoint.checkpoint(func, *inputs, use_reentrant=False)
    return func(*inputs)


def sdpa_attention(self, qkv):
    """ADM's legacy attention through SDPA with 4-D (batch, heads, tokens, channels) inputs, so the fused kernels apply."""
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.reshape(bs, self.n_heads, ch, length).transpose(2, 3).contiguous()
               for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(2, 3).reshape(bs, -1, length)


gd_unet.checkpoint = checkpoint
gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)


def make_ctx(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=False,
                     use_scale_shift_norm=True, use_checkpoint=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False)
    gains = []
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            g = nn.Parameter(torch.ones(mod.out_channels, device=device))
            mod.register_forward_hook(lambda module, inp, out, g=g: out * g.view(1, -1, 1, 1).to(out.dtype))
            gains.append(g)
    norms = [p for mod in m.modules() if isinstance(mod, nn.GroupNorm) for p in (mod.weight, mod.bias)]
    for p in norms:
        p.requires_grad_(True)
    return dict(dev=device, adm=m, gains=gains, norms=norms, norms0=[p.detach().clone() for p in norms])


CTX = [make_ctx(d) for d in DEVICES]
ADM_STATE.clear()


@torch.no_grad()
def reset(ctx):
    for g in ctx["gains"]:
        g.fill_(1.0)
    for p, v in zip(ctx["norms"], ctx["norms0"]):
        p.copy_(v)

## 2 · Estimators: one-step posterior mean (whole or tiled, ×2 flip) and R2R tuning

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()


def matched(sigma01):
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


def tweedie(model, y, sigma01):
    t, ab = matched(sigma01)
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = math.sqrt(ab) * (2 * yp - 1)
    with torch.autocast("cuda", dtype=torch.float16):
        eps = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3]
    return (((x - math.sqrt(1 - ab) * eps.float()) / math.sqrt(ab) + 1) / 2)[..., :h, :w]


def tiled(fn, y, tile=384, overlap=96):
    _, _, H, W = y.shape
    stride = tile - overlap
    cover = lambda n: max(tile, tile + math.ceil(max(n - tile, 0) / stride) * stride)
    yp = F.pad(y, (0, cover(W) - W, 0, cover(H) - H), mode="reflect")
    r = torch.arange(tile, dtype=torch.float32, device=y.device)
    ramp = torch.clamp(torch.minimum(r + 1, tile - r) / overlap, max=1)
    w2 = ramp[:, None] * ramp[None, :]
    acc, wsum = torch.zeros_like(yp), torch.zeros_like(yp[:, :1])
    for i in range(0, yp.shape[-2] - tile + 1, stride):
        for j in range(0, yp.shape[-1] - tile + 1, stride):
            acc[..., i:i + tile, j:j + tile] += fn(yp[..., i:i + tile, j:j + tile]) * w2
            wsum[..., i:i + tile, j:j + tile] += w2
    return (acc / wsum)[..., :H, :W]


@torch.no_grad()
def estimate(ctx, y, sigma, mode):
    """×1 and ×2 (horizontal flip) one-step estimates of y [1,3,H,W], whole image or 384-px tiles."""
    one = lambda z: tweedie(ctx["adm"], z, sigma / 255)
    run = (lambda z: one(z)) if mode == "whole" else (lambda z: tiled(one, z, 384, 96))
    a, b = run(y), run(y.flip(-1)).flip(-1)
    return a, (a + b) / 2


def crops(img, gen, n=BATCH, size=CROP):
    _, H, W = img.shape
    ii = torch.randint(0, H - size + 1, (n,), generator=gen).tolist()
    jj = torch.randint(0, W - size + 1, (n,), generator=gen).tolist()
    return torch.stack([img[:, i:i + size, j:j + size] for i, j in zip(ii, jj)])


def r2r_tune(ctx, noisy, sigma, seed):
    dev, s = ctx["dev"], sigma / 255
    opt = Muon(ctx["gains"] + ctx["norms"], lr=LR, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda")
    gen, ngen = torch.Generator().manual_seed(seed), torch.Generator(device=dev).manual_seed(77_000 + seed)
    for _ in range(STEPS):
        c = crops(noisy, gen)
        z = torch.randn(c.shape, generator=ngen, device=dev)
        loss = F.mse_loss(tweedie(ctx["adm"], c + ALPHA * s * z, s * math.sqrt(1 + ALPHA ** 2)), c - s * z / ALPHA)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

## 3 · Every image: unadapted, then adapted (same noise, same mode)

In [ ]:
def run_job(ctx, job):
    ds, idx, sigma = job
    name, clean = TEST[ds][idx]
    _, _, primary, secondary = PLAN[ds]
    y = noisy_of(ds, idx, sigma)[None].to(ctx["dev"])
    t0 = time.time()
    rows = []
    reset(ctx)
    for mode in [m for m in (primary, secondary) if m]:
        x1, x2 = estimate(ctx, y, sigma, mode)
        rows += [dict(dataset=ds, image=name, idx=idx, sigma=sigma, mode=mode, adapted=False, flip=f, psnr=psnr(x, clean))
                 for f, x in (("×1", x1), ("×2", x2))]
    r2r_tune(ctx, y[0], sigma, seed=100_000_000 * SEED_OFFSET + 10_000 * SEED_SETS.index(ds) + 10 * sigma + idx)
    for mode in [m for m in (primary, secondary) if m]:
        x1, x2 = estimate(ctx, y, sigma, mode)
        rows += [dict(dataset=ds, image=name, idx=idx, sigma=sigma, mode=mode, adapted=True, flip=f, psnr=psnr(x, clean))
                 for f, x in (("×1", x1), ("×2", x2))]
    reset(ctx)
    for r in rows:
        r["seconds"] = time.time() - t0
    return rows


def worker_pool(jobs, fn):
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))
                torch.cuda.empty_cache()

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    t0, last = time.time(), time.time()
    while any(th.is_alive() for th in threads):
        time.sleep(20)
        if time.time() - last > 900:
            last = time.time()
            print(f"  {len(results)}/{len(jobs)} done, {len(failures)} failed, {(time.time() - t0) / 60:.1f} min", flush=True)
            pd.DataFrame([r for rows in results for r in rows]).to_csv(f"{OUT}/scores_partial.csv", index=False)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
jobs = JOBS
res, fail = worker_pool(jobs, run_job)
TIMES["adaptation"] = time.time() - t0
SCORES = pd.DataFrame([r for rows in res for r in rows])
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
print(f"{len(res)}/{len(jobs)} jobs, {len(fail)} failed, {TIMES['adaptation'] / 60:.1f} min")

## 4 · Pre-registered tests and the SOTA check

In [ ]:
def sign_flip_p(d, n_flip=200_000, seed=0):
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1.0, 1.0], size=(n_flip, len(d)))
    return float((np.abs((signs * d).mean(1)) >= abs(d.mean()) - 1e-12).mean())


rows = []
for (ds, sigma, mode), g in SCORES[SCORES.flip == "×2"].groupby(["dataset", "sigma", "mode"]):
    w = g.pivot_table(index="idx", columns="adapted", values="psnr")
    parts = [("confirm-88", w)]
    for split, ww in parts:
        d = (ww[True] - ww[False]).to_numpy()
        rows.append(dict(dataset=ds, sigma=sigma, mode=mode, split=split, primary=mode == PLAN[ds][2], n=len(d),
                         unadapted=ww[False].mean(), adapted=ww[True].mean(), gain=d.mean(),
                         worse_005=float((d < -0.05).mean()), p=sign_flip_p(d)))
T = pd.DataFrame(rows)
# The test is σ 15 on confirm-88 only: dev-12's σ 15 scores were seen when the tile size was chosen (Phase-1 extreme critique).
T["is_test"] = T.primary
prim = T[T.is_test].sort_values("p").copy()
prim["p_holm"] = np.minimum(1.0, np.maximum.accumulate(prim.p.to_numpy() * (len(prim) - np.arange(len(prim)))))
T = T.merge(prim[["dataset", "sigma", "mode", "split", "p_holm"]], on=["dataset", "sigma", "mode", "split"], how="left")
T["success"] = T.is_test & (T.gain >= 0.05) & (T.p_holm < 0.01) & (T.worse_005 <= 0.05)
T["published_best"] = [PUBLISHED.get((d, s)) for d, s in zip(T.dataset, T.sigma)]
# Published numbers are full-100 means; a confirm-88 mean is not comparable with them, so no SOTA flag is computed here.
print(T.round(4).to_string(index=False))
T.to_csv(f"{OUT}/tests.csv", index=False)
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, steps=STEPS, tests=T.to_dict("records"),
                   failed=[j for j, _ in fail]), f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))